# G1 接地報酬の時間的許容幅：P0／P1／P2 追加学習

**資料：**「Unitree G1 接地報酬に許容幅を導入する提案」（2026年10月24日）に基づく主実験．

- **P0**：従来の二値接地報酬（許容幅0 ms）
- **P1**：予定接地・離地時刻の前後 **±20 ms** だけ不一致の減点を滑らかにする
- **P2**：予定接地・離地時刻の前後 **±40 ms** だけ不一致の減点を滑らかにする

3条件で **周期0.8秒，左右位相差0.5，支持率0.55，位相sin/cos入力，報酬係数0.18，PPO構成は変えない**．共通の `model_6500.pt` から各200反復追加学習し，P0/1/2を同じ条件で評価する．

**重要：** これは「周期を可変にする」「位相入力を消す」実験ではありません．接地・離地の予定時刻に対する**報酬の許容幅**だけを変更します．

**Colab：A100推奨**．学習用4096並列環境．新しいDriveフォルダーに出力．前回の学習済みファイル・実験結果は上書きしません．

操作：上から順に実行．学習のセルP0→P1→P2は時間がかかるため，各条件完了を確認しながら進めてください．

## 1．Google Drive とGPUの準備

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, platform, sys, os, json, time, shutil
from datetime import datetime, timezone

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Colab Notebooks/PhysicalAI応用編１/G1_RL_Training')
assert DRIVE.is_dir(), f'前回の学習フォルダーが見つかりません: {DRIVE}'
WORK = Path('/content')
EXP_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
EXP_DIR = DRIVE / f'contact_tolerance_P0_P1_P2_from6500_{EXP_ID}'
EXP_DIR.mkdir(parents=True, exist_ok=False)
print('新しい実験の保存先:', EXP_DIR)
subprocess.run(['nvidia-smi'], check=True)
print('Colab Python:', sys.version.split()[0])

## 2．公式コードを以前と同じコミットに固定

In [ ]:
from pathlib import Path
import subprocess, json

repo = WORK / 'unitree_rl_gym'
rsl = WORK / 'rsl_rl'
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/unitreerobotics/unitree_rl_gym.git', str(repo)], check=True)
if not rsl.exists():
    subprocess.run(['git', 'clone', 'https://github.com/leggedrobotics/rsl_rl.git', str(rsl)], check=True)
# 前回の学習と同じソースコードを再現するためコミットを固定する
subprocess.run(['git', '-C', str(rsl), 'checkout', '2ad79cf0caa85b91721abfe358105f869a784121'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '276801e46c5d433564f24658bac64f254b7d2d4b'], check=True)

rev = {
 'unitree_rl_gym': subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip(),
 'rsl_rl': subprocess.check_output(['git','-C',str(rsl),'rev-parse','HEAD'], text=True).strip()
}
(EXP_DIR/'source_revisions.json').write_text(json.dumps(rev, indent=2), encoding='utf-8')
print(json.dumps(rev, indent=2))

files = [
 repo/'legged_gym/envs/g1/g1_env.py',
 repo/'legged_gym/envs/g1/g1_config.py',
 repo/'legged_gym/envs/base/legged_robot_config.py',
 repo/'legged_gym/scripts/train.py',
]
for f in files:
    print(('OK  ' if f.exists() else 'MISSING '), f.relative_to(repo))
assert all(f.exists() for f in files), '公式リポジトリの構成が変更されています．'

## 3．Isaac Gym配布物とPython 3.8環境の準備

In [ ]:
ARCHIVE = DRIVE / 'IsaacGym_Preview_4_Package.tar'
print('期待するファイル:', ARCHIVE)
if not ARCHIVE.is_file():
    print('ファイル未配置です．公式配布サイトから入手したtarファイルをGoogle Driveへアップロードしてください．')
else:
    print('検出:', ARCHIVE.name, '容量 GiB:', round(ARCHIVE.stat().st_size / 1024**3, 2))

In [ ]:
from pathlib import Path
import subprocess, os

MAMBA_BIN = Path('/content/bin/micromamba')
if not MAMBA_BIN.exists():
    subprocess.run(['bash','-lc',
      'mkdir -p /content && curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'], check=True)
PY38 = Path('/content/g1-python38/bin/python')
env = os.environ.copy()
env['MAMBA_ROOT_PREFIX'] = '/content/mamba-root'
if not PY38.exists():
    subprocess.run([str(MAMBA_BIN), 'create','-y','-p','/content/g1-python38',
         '-c','conda-forge','python=3.8','pip'], check=True, env=env)
print('Python 3.8:', subprocess.check_output([str(PY38), '--version'], text=True).strip())

In [ ]:
from pathlib import Path
import subprocess, os

assert ARCHIVE.is_file(), f'Isaac Gym配布物がありません: {ARCHIVE}'
assert PY38.exists(), '先にPython 3.8環境を作成してください．'

EXTRACT = Path('/content/isaacgym_dist')
if not EXTRACT.exists():
    EXTRACT.mkdir(parents=True)
    subprocess.run(['tar','-xf',str(ARCHIVE),'-C',str(EXTRACT)], check=True)

isaac_python_dirs = [p for p in EXTRACT.rglob('python') if (p/'setup.py').is_file() and p.parent.name == 'isaacgym']
assert isaac_python_dirs, '配布物から isaacgym/python/setup.py が見つかりません．構成を確認してください．'
ISAAC_PY_DIR = isaac_python_dirs[0]
print('Isaac Gym Python package:', ISAAC_PY_DIR)

def pip(*args):
    subprocess.run([str(PY38),'-m','pip',*map(str,args)],check=True)

pip('install','--upgrade','pip','setuptools','wheel')
pip('install','numpy==1.23.5','gym==0.23.1','imageio','ninja','tensorboard')
pip('install','torch==2.3.1','torchvision==0.18.1','torchaudio==2.3.1',
    '--index-url','https://download.pytorch.org/whl/cu121')
# 既存GPU版PyTorchを維持し，旧Isaac Gymインストールの依存関係で置き換えない
pip('install','-e',str(ISAAC_PY_DIR),'--no-deps')
pip('install','-e',str(rsl))
pip('install','-e',str(repo))
print('依存ライブラリのインストールが終了しました．まだGPU動作の確認は済んでいません．')

## 4．GPU・PyTorch・Ninjaの確認

In [ ]:
import subprocess, os
RUN_ENV = os.environ.copy()
RUN_ENV['LD_LIBRARY_PATH'] = '/content/g1-python38/lib:' + RUN_ENV.get('LD_LIBRARY_PATH','')
RUN_ENV['TORCH_EXTENSIONS_DIR'] = '/content/g1_torch_extensions'
RUN_ENV['PATH'] = str(PY38.parent) + os.pathsep + RUN_ENV.get('PATH', '')
Path(RUN_ENV['TORCH_EXTENSIONS_DIR']).mkdir(exist_ok=True)

probe = r"""
from isaacgym import gymapi
import torch
print('IsaacGym import OK')
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDAを認識していません')
"""
p = subprocess.run([str(PY38),'-c',probe],env=RUN_ENV,capture_output=True,text=True)
print(p.stdout)
print(p.stderr[-6000:])
assert p.returncode == 0, '互換性／依存関係の問題が発生しました．ここで停止してください．'
(EXP_DIR/'python38_freeze.txt').write_text(subprocess.check_output(
    [str(PY38),'-m','pip','freeze'],text=True,env=RUN_ENV),encoding='utf-8')

In [ ]:
import os
import subprocess
from pathlib import Path

PY38 = Path('/content/g1-python38/bin/python')
BIN38 = PY38.parent

# Ninjaが未インストールの場合も対応
subprocess.run(
    [str(PY38), '-m', 'pip', 'install', 'ninja'],
    check=True
)

# 試験学習の子プロセスに渡す環境変数を修正
RUN_ENV['PATH'] = str(BIN38) + os.pathsep + RUN_ENV.get('PATH', '')

# 子プロセスからNinjaが見えるか確認
check = subprocess.run(
    [
        str(PY38), '-c',
        'import shutil; '
        'from torch.utils.cpp_extension import is_ninja_available; '
        'print("Ninja path:", shutil.which("ninja")); '
        'print("Ninja available:", is_ninja_available())'
    ],
    env=RUN_ENV,
    capture_output=True,
    text=True
)

print(check.stdout)
print(check.stderr[-2000:])
assert check.returncode == 0
assert 'Ninja available: True' in check.stdout

## 5．実験条件と公式報酬を確認

公式 `_reward_contact()` は片足につき予定接地状態と実際の接地状態の一致で1，不一致で0を返す．
原本は変更せず，**P1/P2だけ実行時にメソッドを差し替える**．P0は公式コードをそのまま呼ぶ．

In [ ]:
from pathlib import Path
import subprocess, json, re, hashlib
G1_FILE = repo / 'legged_gym/envs/g1/g1_env.py'
CFG_FILE = repo / 'legged_gym/envs/g1/g1_config.py'
env_src = G1_FILE.read_text(encoding='utf-8')
cfg_src = CFG_FILE.read_text(encoding='utf-8')
assert re.search(r'period\s*=\s*0\.8\b', env_src), '0.8秒周期が見つかりません'
assert 'sin_phase' in env_src and 'cos_phase' in env_src
assert re.search(r'^\s*contact\s*=\s*0\.18\b', cfg_src, re.M)
assert '_reward_contact' in env_src
assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip() == '276801e46c5d433564f24658bac64f254b7d2d4b'
assert subprocess.check_output(['git','-C',str(rsl),'rev-parse','HEAD'],text=True).strip() == '2ad79cf0caa85b91721abfe358105f869a784121'
print('公式コード確認：0.8 s位相入力，接地報酬0.18，git固定 OK')
assert 'res += ~(contact ^ is_stance)' in env_src, '想定している公式二値接地報酬と異なります'
assert 'self.leg_phase[:, i] < 0.55' in env_src
assert 'self.contact_forces[:, self.feet_indices[i], 2] > 1' in env_src
print('報酬関数が想定コードと一致しています．')

## 6．P0／P1／P2用の学習スクリプトを生成

許容幅 \(\delta\) は秒単位．\(\delta=0\) は公式コードをそのまま使用．

脚位相 \(\phi_i\) について切替え位相は **0と0.55**．これらへの循環距離を \(d_i\) とし，

\[x_i=\mathrm{clip}\left(d_i/(\delta/0.8),0,1
ight),\quad w_i=x_i^2(3-2x_i)\]

\[r_\mathrm{contact}=\sum_i [1-w_i\,\mathbf1(c_i
e s_i)]\]

を適用します．**報酬全体を弱めるわけではありません．**

In [ ]:
TRAIN_SCRIPT = Path('/content/g1_contact_tolerance_train.py')
TRAIN_SCRIPT.write_text("# Isaac Gym must be imported before torch\nfrom isaacgym import gymapi\nfrom legged_gym.envs import *\nfrom legged_gym.envs.g1.g1_env import G1Robot\nfrom legged_gym.utils import get_args, task_registry\nimport argparse, sys, torch\n\nOFFICIAL_CONTACT_REWARD = G1Robot._reward_contact\nPERIOD_S = 0.8\nSUPPORT_FRACTION = 0.55\n\ndef make_tolerant_contact_reward(tolerance_s):\n    if tolerance_s == 0:\n        return OFFICIAL_CONTACT_REWARD\n    delta_phase = tolerance_s / PERIOD_S\n    assert 0.0 < delta_phase < .1, delta_phase\n    def reward(self):\n        # leg_phase [N,2] = each foot's local phase in [0,1)\n        # planned contact = local phase < 0.55\n        # closest periodic transition: 0.0 and 0.55\n        p = self.leg_phase\n        stance = p < SUPPORT_FRACTION\n        cp0 = torch.minimum(p, 1.0-p)\n        cp1_abs = torch.abs(p-SUPPORT_FRACTION)\n        cp1 = torch.minimum(cp1_abs, 1.0-cp1_abs)\n        dist = torch.minimum(cp0, cp1)\n        x = torch.clamp(dist / delta_phase, min=0.0, max=1.0)\n        weight = x*x*(3.0-2.0*x)  # smoothstep, 0 at transition, 1 outside window\n        contact = self.contact_forces[:, self.feet_indices, 2] > 1.0\n        mismatch = (contact ^ stance).float()\n        reward_feet = 1.0 - weight * mismatch\n        return reward_feet.sum(dim=1)\n    return reward\n\ndef reward_self_test():\n    from types import SimpleNamespace\n    phase = torch.tensor([\n        [0.55+0.01/0.8, 0.25], # left: 10ms after stance end, but still in contact\n        [0.55+0.10/0.8, 0.25], # left: far outside tolerance\n        [0.00, 0.25],          # left: exact planned transition\n    ], dtype=torch.float32)\n    forces = torch.zeros((3,2,3), dtype=torch.float32)\n    forces[:,0,2] = 10.0  # left: contact (mismatch for rows 0,1)\n    forces[:,1,2] = 10.0  # right: contact (match)\n    dummy = SimpleNamespace(num_envs=3, feet_num=2, device='cpu',\n                            feet_indices=torch.tensor([0,1]),\n                            contact_forces=forces, leg_phase=phase)\n    official = OFFICIAL_CONTACT_REWARD(dummy)\n    zero = make_tolerant_contact_reward(0.0)(dummy)\n    assert torch.equal(zero,official), (official,zero)\n    assert abs(float(official[0])-1.0)<1e-6\n    p1=make_tolerant_contact_reward(0.020)(dummy)\n    p2=make_tolerant_contact_reward(0.040)(dummy)\n    # at 10ms: P1 half penalty -> 1.5 total, P2 0.15625 penalty -> 1.84375\n    assert abs(float(p1[0])-1.5)<1e-5, p1\n    assert abs(float(p2[0])-1.84375)<1e-5, p2\n    assert abs(float(p2[1])-1.)<1e-5, p2\n    # exact planned contact transition must be accepted (row 2, no mismatch)\n    assert abs(float(p2[2])-2.)<1e-5, p2\n    print('Reward tests OK: P0 exact official / P1 10ms=1.5 / P2 10ms=1.84375 / far=1',flush=True)\n\nif __name__=='__main__':\n    ap=argparse.ArgumentParser(add_help=False)\n    ap.add_argument('--tolerance_ms', type=int, required=False, default=0)\n    ap.add_argument('--expected_start', type=int, required=False, default=6500)\n    ap.add_argument('--self_test',action='store_true')\n    cfg_args,rest=ap.parse_known_args()\n    reward_self_test()\n    if cfg_args.self_test:\n        sys.exit(0)\n    assert cfg_args.tolerance_ms in (0,20,40)\n    sys.argv=[sys.argv[0]]+rest\n    args=get_args()\n    assert args.task=='g1' and args.resume and args.headless\n    cfg,training_cfg=task_registry.get_cfgs(name='g1')\n    cfg.seed=int(args.seed)\n    original_weight=float(cfg.rewards.scales.contact)\n    assert abs(original_weight-0.18)<1e-7\n    tolerance_seconds=cfg_args.tolerance_ms/1000.0\n    if cfg_args.tolerance_ms:\n        G1Robot._reward_contact=make_tolerant_contact_reward(tolerance_seconds)\n    else:\n        G1Robot._reward_contact=OFFICIAL_CONTACT_REWARD\n    assert float(cfg.rewards.scales.contact)==0.18\n    print('PPO training: tolerance_ms',cfg_args.tolerance_ms,'contact_weight',original_weight,flush=True)\n    print('Phase period=0.8s / left-right offset=0.5 / stance=0.55 / sin-cos inputs unchanged',flush=True)\n    env,_=task_registry.make_env(name='g1', args=args, env_cfg=cfg)\n    runner,training_cfg=task_registry.make_alg_runner(env=env, name='g1',args=args,train_cfg=training_cfg)\n    assert runner.current_learning_iteration==cfg_args.expected_start, runner.current_learning_iteration\n    assert runner.save_interval==50, runner.save_interval\n    print('Starting iteration:',runner.current_learning_iteration,'additional:',training_cfg.runner.max_iterations,flush=True)\n    runner.learn(num_learning_iterations=training_cfg.runner.max_iterations,init_at_random_ep_len=True)\n", encoding='utf-8')
compile("# Isaac Gym must be imported before torch\nfrom isaacgym import gymapi\nfrom legged_gym.envs import *\nfrom legged_gym.envs.g1.g1_env import G1Robot\nfrom legged_gym.utils import get_args, task_registry\nimport argparse, sys, torch\n\nOFFICIAL_CONTACT_REWARD = G1Robot._reward_contact\nPERIOD_S = 0.8\nSUPPORT_FRACTION = 0.55\n\ndef make_tolerant_contact_reward(tolerance_s):\n    if tolerance_s == 0:\n        return OFFICIAL_CONTACT_REWARD\n    delta_phase = tolerance_s / PERIOD_S\n    assert 0.0 < delta_phase < .1, delta_phase\n    def reward(self):\n        # leg_phase [N,2] = each foot's local phase in [0,1)\n        # planned contact = local phase < 0.55\n        # closest periodic transition: 0.0 and 0.55\n        p = self.leg_phase\n        stance = p < SUPPORT_FRACTION\n        cp0 = torch.minimum(p, 1.0-p)\n        cp1_abs = torch.abs(p-SUPPORT_FRACTION)\n        cp1 = torch.minimum(cp1_abs, 1.0-cp1_abs)\n        dist = torch.minimum(cp0, cp1)\n        x = torch.clamp(dist / delta_phase, min=0.0, max=1.0)\n        weight = x*x*(3.0-2.0*x)  # smoothstep, 0 at transition, 1 outside window\n        contact = self.contact_forces[:, self.feet_indices, 2] > 1.0\n        mismatch = (contact ^ stance).float()\n        reward_feet = 1.0 - weight * mismatch\n        return reward_feet.sum(dim=1)\n    return reward\n\ndef reward_self_test():\n    from types import SimpleNamespace\n    phase = torch.tensor([\n        [0.55+0.01/0.8, 0.25], # left: 10ms after stance end, but still in contact\n        [0.55+0.10/0.8, 0.25], # left: far outside tolerance\n        [0.00, 0.25],          # left: exact planned transition\n    ], dtype=torch.float32)\n    forces = torch.zeros((3,2,3), dtype=torch.float32)\n    forces[:,0,2] = 10.0  # left: contact (mismatch for rows 0,1)\n    forces[:,1,2] = 10.0  # right: contact (match)\n    dummy = SimpleNamespace(num_envs=3, feet_num=2, device='cpu',\n                            feet_indices=torch.tensor([0,1]),\n                            contact_forces=forces, leg_phase=phase)\n    official = OFFICIAL_CONTACT_REWARD(dummy)\n    zero = make_tolerant_contact_reward(0.0)(dummy)\n    assert torch.equal(zero,official), (official,zero)\n    assert abs(float(official[0])-1.0)<1e-6\n    p1=make_tolerant_contact_reward(0.020)(dummy)\n    p2=make_tolerant_contact_reward(0.040)(dummy)\n    # at 10ms: P1 half penalty -> 1.5 total, P2 0.15625 penalty -> 1.84375\n    assert abs(float(p1[0])-1.5)<1e-5, p1\n    assert abs(float(p2[0])-1.84375)<1e-5, p2\n    assert abs(float(p2[1])-1.)<1e-5, p2\n    # exact planned contact transition must be accepted (row 2, no mismatch)\n    assert abs(float(p2[2])-2.)<1e-5, p2\n    print('Reward tests OK: P0 exact official / P1 10ms=1.5 / P2 10ms=1.84375 / far=1',flush=True)\n\nif __name__=='__main__':\n    ap=argparse.ArgumentParser(add_help=False)\n    ap.add_argument('--tolerance_ms', type=int, required=False, default=0)\n    ap.add_argument('--expected_start', type=int, required=False, default=6500)\n    ap.add_argument('--self_test',action='store_true')\n    cfg_args,rest=ap.parse_known_args()\n    reward_self_test()\n    if cfg_args.self_test:\n        sys.exit(0)\n    assert cfg_args.tolerance_ms in (0,20,40)\n    sys.argv=[sys.argv[0]]+rest\n    args=get_args()\n    assert args.task=='g1' and args.resume and args.headless\n    cfg,training_cfg=task_registry.get_cfgs(name='g1')\n    cfg.seed=int(args.seed)\n    original_weight=float(cfg.rewards.scales.contact)\n    assert abs(original_weight-0.18)<1e-7\n    tolerance_seconds=cfg_args.tolerance_ms/1000.0\n    if cfg_args.tolerance_ms:\n        G1Robot._reward_contact=make_tolerant_contact_reward(tolerance_seconds)\n    else:\n        G1Robot._reward_contact=OFFICIAL_CONTACT_REWARD\n    assert float(cfg.rewards.scales.contact)==0.18\n    print('PPO training: tolerance_ms',cfg_args.tolerance_ms,'contact_weight',original_weight,flush=True)\n    print('Phase period=0.8s / left-right offset=0.5 / stance=0.55 / sin-cos inputs unchanged',flush=True)\n    env,_=task_registry.make_env(name='g1', args=args, env_cfg=cfg)\n    runner,training_cfg=task_registry.make_alg_runner(env=env, name='g1',args=args,train_cfg=training_cfg)\n    assert runner.current_learning_iteration==cfg_args.expected_start, runner.current_learning_iteration\n    assert runner.save_interval==50, runner.save_interval\n    print('Starting iteration:',runner.current_learning_iteration,'additional:',training_cfg.runner.max_iterations,flush=True)\n    runner.learn(num_learning_iterations=training_cfg.runner.max_iterations,init_at_random_ep_len=True)\n", str(TRAIN_SCRIPT), 'exec')
print('学習スクリプト作成:', TRAIN_SCRIPT)

## 7．報酬関数の単体テスト（学習前に必ず実行）

In [ ]:
import subprocess
p=subprocess.run([str(PY38),str(TRAIN_SCRIPT),'--self_test'],
                 cwd=str(repo),env=RUN_ENV,text=True,capture_output=True)
print(p.stdout)
print(p.stderr[-3000:])
assert p.returncode==0,'接地報酬の単体テストに失敗．学習を開始しないでください．'

## 8．6500反復モデルの作業用コピーを作成

学習済み重みとPPO optimizer状態は維持し，ファイル名と内部iterのずれだけを作業用コピーで修正します．原本はSHA256で保護．

In [ ]:
import json, shutil, subprocess, hashlib
SOURCE_MODEL = DRIVE / 'full_logs/g1/Oct23_16-40-56_colab_continue_to_10000/model_6500.pt'
assert SOURCE_MODEL.is_file() and SOURCE_MODEL.stat().st_size > 100000, SOURCE_MODEL
SRC_DIGEST = hashlib.sha256(SOURCE_MODEL.read_bytes()).hexdigest()
SEED_RUN = 'phase_seed_6500_'+EXP_ID
LOCAL_SEED_DIR = repo / 'logs/g1' / SEED_RUN
LOCAL_SEED_DIR.mkdir(parents=True, exist_ok=False)
SEED_MODEL = LOCAL_SEED_DIR / 'model_6500.pt'
# CPUで読み出して保存すると内容が同一になる；checkpoint内のパラメータと optimizer は維持する
fix_code = r'''import torch, sys
source,destination=sys.argv[1],sys.argv[2]
x=torch.load(source,map_location='cpu')
assert set(['model_state_dict','optimizer_state_dict','iter']).issubset(x)
print('元の内部iter:',x['iter'],flush=True)
assert int(x['iter'])==1000
x['iter']=6500
torch.save(x,destination)
print('作業用コピーの内部iter:',x['iter'],flush=True)
'''
p = subprocess.run([str(PY38),'-c',fix_code,str(SOURCE_MODEL),str(SEED_MODEL)],
                   cwd=str(repo), env=RUN_ENV, text=True, capture_output=True)
print(p.stdout, p.stderr[-3000:]); assert p.returncode == 0
assert hashlib.sha256(SOURCE_MODEL.read_bytes()).hexdigest() == SRC_DIGEST
print('原本のSHA256不変を確認:',SRC_DIGEST)
(EXP_DIR/'experiment_plan.json').write_text(json.dumps({
  'base_checkpoint':str(SOURCE_MODEL), 'base_sha256':SRC_DIGEST,
  'base_model_iter':6500, 'num_envs':4096,
  'additional_iterations':200, 'seed':1, 'start_iteration':6500,
  'contact_reward_weight':0.18, 'tolerance_ms':{'P0':0,'P1':20,'P2':40},
  'period_s':0.8,'phase_observations_modified':False, 'support_fraction':0.55, 'left_right_phase_offset':0.5,
  'unitree_rl_gym_commit':'276801e46c5d433564f24658bac64f254b7d2d4b',
  'rsl_rl_commit':'2ad79cf0caa85b91721abfe358105f869a784121'},
  indent=2, ensure_ascii=False),encoding='utf-8')

## 9．追加学習の共通条件

In [ ]:
ADDITIONAL_ITERATIONS=200
NUM_ENVS=4096
SEED=1
TOLERANCES_MS={'P0':0,'P1':20,'P2':40}
FINAL_ITER=6500+ADDITIONAL_ITERATIONS
assert len(TOLERANCES_MS)==3
for label,ms in TOLERANCES_MS.items():
    print(f'{label}: tolerance ±{ms} ms / weight 0.18 / 6500→{FINAL_ITER}')

In [ ]:
import time, shutil, subprocess, os, re
from pathlib import Path

FINAL_ITER = 6500+ADDITIONAL_ITERATIONS
LOCAL_ROOT = repo / 'logs/g1'
BRANCHES = {label: {'tolerance_ms': ms, 'run_name': f'g1_tol_{label}_{EXP_ID}'} for label,ms in TOLERANCES_MS.items()}

def local_branch_run(label):
    candidates=list(LOCAL_ROOT.glob('*_'+BRANCHES[label]['run_name']))
    if len(candidates)>1: raise RuntimeError('重複する学習フォルダーが存在します')
    return candidates[0] if candidates else None

def sync_branch(label):
    origin=local_branch_run(label)
    if origin is None: return []
    dest=EXP_DIR / label / 'checkpoints'
    dest.mkdir(parents=True, exist_ok=True)
    copied=[]
    for f in sorted(origin.glob('model_*.pt')):
        if not f.is_file() or f.stat().st_size < 100000: continue
        target=dest/f.name
        if target.is_file() and target.stat().st_size == f.stat().st_size: continue
        tmp=target.with_suffix('.pt.partial')
        shutil.copy2(f,tmp)
        tmp.replace(target)
        copied.append(f.name)
    # Tensorboardは終了後に完全版を保存
    return copied

def run_branch(label):
    conf=BRANCHES[label]
    branch_dest=EXP_DIR/label
    branch_dest.mkdir(parents=True,exist_ok=True)
    if (branch_dest/'FINISHED.ok').exists():
        print(label,'は実行済みです．再学習はしません．');return
    assert local_branch_run(label) is None, '同じ名前の学習が既に開始されています．誤って二重実行しないよう確認してください．'
    cmd=[str(PY38),'-u',str(TRAIN_SCRIPT),'--task=g1','--headless',
         '--sim_device=cuda:0','--rl_device=cuda:0',
         f'--num_envs={NUM_ENVS}',f'--max_iterations={ADDITIONAL_ITERATIONS}',
         f'--seed={SEED}','--resume',f'--load_run={SEED_RUN}',
         '--checkpoint=6500',f'--run_name={conf["run_name"]}',
         '--tolerance_ms',str(conf['tolerance_ms']),'--expected_start','6500']
    log=branch_dest/'train_stdout.txt'
    env=RUN_ENV.copy()
    env['PATH']=str(PY38.parent)+os.pathsep+env.get('PATH','')
    print('実行:',label,'tolerance_ms=',conf['tolerance_ms'],'保存先=',branch_dest,flush=True)
    with log.open('w',encoding='utf-8') as f:
        proc=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT,
                              cwd=str(repo),env=env)
        try:
            while proc.poll() is None:
                time.sleep(45)
                copied=sync_branch(label)
                if copied:print('途中checkpointを保存:',copied,flush=True)
                tail=log.read_text(encoding='utf-8',errors='replace')[-850:]
                print(label,'実行中｜', tail.splitlines()[-1:] or ['ログ準備中'],flush=True)
        except KeyboardInterrupt:
            proc.terminate(); proc.wait(timeout=30); sync_branch(label)
            print('中断しました．既存の中間checkpointはDriveに保存しました．')
            raise
    copied=sync_branch(label)
    print('完了コード:',proc.returncode,'最終保存:',copied,flush=True)
    if proc.returncode:
        print(log.read_text(encoding='utf-8',errors='replace')[-5000:])
        raise RuntimeError(f'{label}の学習が失敗しました．ログを確認してください：{log}')
    expected=branch_dest/'checkpoints'/f'model_{FINAL_ITER}.pt'
    assert expected.exists(),f'最終チェックポイントがありません: {expected}'
    run=local_branch_run(label)
    for f in run.glob('events.out.tfevents.*'):
        shutil.copy2(f,branch_dest/f.name)
    (branch_dest/'FINISHED.ok').write_text('Completed '+str(FINAL_ITER),encoding='utf-8')
    print(f'{label} 最終モデル:', expected)

## 100．P0：許容幅 ±0 ms の追加学習

In [ ]:
run_branch('P0')

## 101．P1：許容幅 ±20 ms の追加学習

In [ ]:
assert (EXP_DIR/'P0/FINISHED.ok').exists(), '先にP0の学習を完了してください．'
run_branch('P1')

## 102．P2：許容幅 ±40 ms の追加学習

In [ ]:
assert (EXP_DIR/'P1/FINISHED.ok').exists(), '先にP1の学習を完了してください．'
run_branch('P2')

## 11．チェックポイントと学習ログの確認

In [ ]:
for label in TOLERANCES_MS:
    ck=EXP_DIR/label/'checkpoints'/f'model_{FINAL_ITER}.pt'
    done=EXP_DIR/label/'FINISHED.ok'
    print(label, 'checkpoint:',ck.is_file(),'done:',done.is_file(),'size MiB:',round(ck.stat().st_size/2**20,2) if ck.is_file() else None)
    assert done.is_file() and ck.is_file()
print('3条件すべての追加学習が終了しました．')

## 12．P0・P1・P2と6500基準方策の共通評価スクリプト

対照条件も含め全方策を**同じシミュレーター設定と指令速度**で実行．
接地イベントは起動直後から記録し，定常解析には4〜18秒の区間を用いる．鉛直接触力は全試行で保存．

評価時にP1/P2の学習用報酬を適用する必要はありません．学習後のニューラルネットワークだけを評価します．

In [ ]:
EVAL_SCRIPT=Path('/content/g1_contact_tolerance_eval.py')
EVAL_SCRIPT.write_text("import argparse\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom isaacgym import gymapi  # torch より前にロードすること（Isaac Gym）\nimport torch\nfrom legged_gym.envs import *  # noqa: F401,F403 — g1 を task registry に登録\nfrom legged_gym.utils import task_registry, get_args\n\n\nif __name__ == '__main__':\n    import sys\n    ap = argparse.ArgumentParser(add_help=False)\n    ap.add_argument('--eval_model', required=True)\n    ap.add_argument('--eval_label', required=True)\n    ap.add_argument('--eval_out', required=True)\n    ap.add_argument('--eval_reps', type=int, default=4)\n    ap.add_argument('--eval_duration', type=float, default=18.0)\n    eval_args, unparsed = ap.parse_known_args()\n    sys.argv = [sys.argv[0]] + unparsed\n    # コード再現性を優先し，全方策で同じシード．\n    args = get_args()\n    assert args.task == 'g1'\n\n    speeds = [0.2, 0.6, 1.0]\n    reps = eval_args.eval_reps\n    n = len(speeds) * reps\n    duration = eval_args.eval_duration\n    assert 0 < duration < 19.5, '評価はタイムアウト20秒より短くしてください．'\n    args.num_envs = n\n    args.seed = 2026\n\n    cfg, train_cfg = task_registry.get_cfgs(name='g1')\n    cfg.env.num_envs = n\n    cfg.seed = args.seed  # reproduce evaluation seed across all models\n    cfg.env.test = False  # シミュレーションと実時間の同期sleepをしない\n    cfg.noise.add_noise = False\n    cfg.domain_rand.randomize_friction = False\n    cfg.domain_rand.randomize_base_mass = False\n    cfg.domain_rand.push_robots = False\n    cfg.terrain.mesh_type = 'plane'\n    cfg.terrain.curriculum = False\n    cfg.commands.curriculum = False\n    cfg.commands.heading_command = False  # 前進v, lateral=0, yaw rate=0を直接指定\n    cfg.commands.resampling_time = 1000000.0  # 評価中の指令更新を止める\n\n    env, _ = task_registry.make_env(name='g1', args=args, env_cfg=cfg)\n    runner, _ = task_registry.make_alg_runner(env=env, name='g1', args=args,\n                                               train_cfg=train_cfg, log_root=None)\n    model = Path(eval_args.eval_model)\n    assert model.is_file(), model\n    checkpoint = torch.load(str(model), map_location=env.device)\n    recorded_iter = checkpoint.get('iter')\n    print('Evaluate label:', eval_args.eval_label,\n          'Internal checkpoint iteration:', recorded_iter, flush=True)\n    assert 'model_state_dict' in checkpoint, 'model_state_dictが見つかりません．' \n    runner.alg.actor_critic.load_state_dict(checkpoint['model_state_dict'])\n    inference = runner.get_inference_policy(device=env.device)\n    # fresh Python process / fresh runner: LSTM hidden stateはNoneから開始\n    assert runner.alg.actor_critic.memory_a.hidden_states is None\n    device = env.device\n    dt = float(env.dt)\n    steps = int(round(duration / dt))\n    assert steps < env.max_episode_length\n\n    # 各速度をreps個並列に割り当てる．4方策とも同一seed，env順で評価．\n    targets = torch.tensor([v for v in speeds for _ in range(reps)],\n                           dtype=torch.float, device=device)\n    # runner初期化でenv.reset()が走っているので，その後に指令を書き換え，観測も再計算する\n    env.commands[:, :] = 0.0\n    env.commands[:, 0] = targets\n    env.compute_observations()\n    obs = env.get_observations()\n    assert obs.shape == (n, 47), obs.shape\n\n    survived = torch.ones(n, device=device, dtype=torch.bool)\n    first_fall = torch.full((n,), float('nan'), device=device)\n    sum_v = torch.zeros(n, device=device)\n    count_v = torch.zeros(n, device=device)\n    sum_v_steady = torch.zeros(n, device=device)\n    sum_e2_steady = torch.zeros(n, device=device)\n    count_steady = torch.zeros(n, device=device)\n    sum_roll2 = torch.zeros(n, device=device)\n    sum_pitch2 = torch.zeros(n, device=device)\n    min_height = torch.full((n,), float('inf'), device=device)\n    trajectory = []\n    contacts_log = []\n    forces_log = []\n    feet = env.feet_indices\n    assert feet.numel() == 2, feet\n    print('Feet indices (L/R):', feet.tolist(), flush=True)\n    previous_contact = (env.contact_forces[:, feet, 2] > 1.0).clone()\n    idx_reps = [k * reps for k in range(len(speeds))]\n\n    print('evaluation:', eval_args.eval_label, 'envs', n, 'steps', steps, 'dt', dt, flush=True)\n    with torch.no_grad():\n        for step in range(steps):\n            t = (step + 1) * dt\n            active = survived.clone()\n            actions = inference(obs.detach())\n            obs, _, _, dones, _ = env.step(actions.detach())\n            # 発生した終了は学習時の転倒条件（骨盤接触または姿勢閾値），タイムアウト前に限定\n            dones = dones.bool()\n            just_fell = active & dones\n            first_fall[just_fell] = t\n            survived &= ~dones\n            runner.alg.actor_critic.memory_a.reset(dones)\n\n            vx = env.base_lin_vel[:, 0]\n            roll = env.rpy[:, 0]\n            pitch = env.rpy[:, 1]\n            height = env.base_pos[:, 2]\n            alive_this_sample = active & (~dones)\n            val = alive_this_sample.float()\n            sum_v += torch.where(alive_this_sample, vx, torch.zeros_like(vx))\n            count_v += val\n            sum_roll2 += torch.where(alive_this_sample, roll ** 2, torch.zeros_like(roll))\n            sum_pitch2 += torch.where(alive_this_sample, pitch ** 2, torch.zeros_like(pitch))\n            min_height = torch.where(alive_this_sample, torch.minimum(min_height, height), min_height)\n            if t >= 6.0:\n                sum_v_steady += torch.where(alive_this_sample, vx, torch.zeros_like(vx))\n                sum_e2_steady += torch.where(alive_this_sample, (vx-targets)**2, torch.zeros_like(vx))\n                count_steady += val\n\n            # 左右接地変化（実験開始から）の記録．力の生値を閾値1 Nで接地判定\n            current_contact = env.contact_forces[:, feet, 2] > 1.0\n            if t >= 0.0:\n                changed = current_contact ^ previous_contact\n                for idx, side_idx in changed.nonzero(as_tuple=False).tolist():\n                    if bool(alive_this_sample[idx].item()):\n                        contacts_log.append({\n                            'model': eval_args.eval_label,\n                            'command_mps': speeds[idx // reps],\n                            'rep': idx % reps, 'side': ('L', 'R')[side_idx],\n                            'event': 'touchdown' if bool(current_contact[idx, side_idx]) else 'liftoff',\n                            'time_s': round(t, 5)\n                        })\n            # All surviving environments: left/right vertical contact force, dt~20ms\n            fz_l = env.contact_forces[:,feet[0],2].detach().cpu().tolist()\n            fz_r = env.contact_forces[:,feet[1],2].detach().cpu().tolist()\n            alive_mask = alive_this_sample.detach().cpu().tolist()\n            for j in range(n):\n                if alive_mask[j]:\n                    forces_log.append({'model':eval_args.eval_label,\n                                       'command_mps':speeds[j//reps], 'rep':j%reps,\n                                       'time_s':round(t,5), 'left_fz_N':fz_l[j],\n                                       'right_fz_N':fz_r[j]})\n            previous_contact = current_contact.clone()\n\n            # 各速度で最初の試行について，軌跡を取得．落ちた後は記録しない\n            for j in idx_reps:\n                if bool(alive_this_sample[j].item()):\n                    trajectory.append({\n                        'label': eval_args.eval_label, 'command_mps': float(targets[j].item()),\n                        'time_s': round(t, 5), 'base_vx_mps': float(vx[j].item()),\n                        'base_height_m': float(height[j].item()),\n                        'roll_rad': float(roll[j].item()),\n                        'pitch_rad': float(pitch[j].item())\n                    })\n            if not bool(survived.any().item()):\n                break\n\n    def cpu_np(x):\n        return x.detach().cpu().numpy()\n\n    ok = cpu_np(survived)\n    falls = cpu_np(first_fall)\n    obs_counts = cpu_np(count_v)\n    steady_counts = cpu_np(count_steady)\n    vmeans = cpu_np(sum_v)\n    vsteady = cpu_np(sum_v_steady)\n    e2 = cpu_np(sum_e2_steady)\n    r2 = cpu_np(sum_roll2)\n    p2 = cpu_np(sum_pitch2)\n    hmin = cpu_np(min_height)\n\n    rows = []\n    for j in range(n):\n        rows.append({\n            'label': eval_args.eval_label,\n            'command_mps': speeds[j // reps],\n            'rep': j % reps,\n            'seed': args.seed,\n            'survived_18s': int(ok[j]),\n            'fall_time_s': round(float(falls[j]), 4) if np.isfinite(falls[j]) else '',\n            'observed_s': round(float(obs_counts[j] * dt), 4),\n            'mean_vx_active_mps': float(vmeans[j] / obs_counts[j]) if obs_counts[j] else '',\n            'mean_vx_steady_mps': float(vsteady[j] / steady_counts[j]) if steady_counts[j] else '',\n            'rmse_vx_steady_mps': float(np.sqrt(e2[j] / steady_counts[j])) if steady_counts[j] else '',\n            'rms_roll_rad': float(np.sqrt(r2[j] / obs_counts[j])) if obs_counts[j] else '',\n            'rms_pitch_rad': float(np.sqrt(p2[j] / obs_counts[j])) if obs_counts[j] else '',\n            'min_base_height_m': float(hmin[j]) if np.isfinite(hmin[j]) else '',\n        })\n\n    dst = Path(eval_args.eval_out)\n    dst.mkdir(parents=True, exist_ok=True)\n    def save_csv(path, data):\n        if not data:\n            return\n        with path.open('w', newline='', encoding='utf-8') as f:\n            writer = csv.DictWriter(f, fieldnames=list(data[0]))\n            writer.writeheader()\n            writer.writerows(data)\n    save_csv(dst / f'trials_{eval_args.eval_label}.csv', rows)\n    save_csv(dst / f'trajectories_{eval_args.eval_label}.csv', trajectory)\n    save_csv(dst / f'foot_events_{eval_args.eval_label}.csv', contacts_log)\n    save_csv(dst / f'forces_{eval_args.eval_label}.csv', forces_log)\n    with (dst / f'config_{eval_args.eval_label}.json').open('w') as f:\n        json.dump({'model': str(model), 'label': eval_args.eval_label,\n                   'speeds_mps': speeds, 'reps_per_speed': reps,\n                   'duration_s': duration, 'seed': args.seed,\n                   'dt_s': dt,\n                   'evaluation': 'fixed x command / yaw=0 / no noise / no push / fixed friction and mass'}, f, indent=2)\n    print('saved:', dst, 'rows:', len(rows), 'survived:', int(np.sum(ok)), '/', n, flush=True)\n\n",encoding='utf-8')
compile("import argparse\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom isaacgym import gymapi  # torch より前にロードすること（Isaac Gym）\nimport torch\nfrom legged_gym.envs import *  # noqa: F401,F403 — g1 を task registry に登録\nfrom legged_gym.utils import task_registry, get_args\n\n\nif __name__ == '__main__':\n    import sys\n    ap = argparse.ArgumentParser(add_help=False)\n    ap.add_argument('--eval_model', required=True)\n    ap.add_argument('--eval_label', required=True)\n    ap.add_argument('--eval_out', required=True)\n    ap.add_argument('--eval_reps', type=int, default=4)\n    ap.add_argument('--eval_duration', type=float, default=18.0)\n    eval_args, unparsed = ap.parse_known_args()\n    sys.argv = [sys.argv[0]] + unparsed\n    # コード再現性を優先し，全方策で同じシード．\n    args = get_args()\n    assert args.task == 'g1'\n\n    speeds = [0.2, 0.6, 1.0]\n    reps = eval_args.eval_reps\n    n = len(speeds) * reps\n    duration = eval_args.eval_duration\n    assert 0 < duration < 19.5, '評価はタイムアウト20秒より短くしてください．'\n    args.num_envs = n\n    args.seed = 2026\n\n    cfg, train_cfg = task_registry.get_cfgs(name='g1')\n    cfg.env.num_envs = n\n    cfg.seed = args.seed  # reproduce evaluation seed across all models\n    cfg.env.test = False  # シミュレーションと実時間の同期sleepをしない\n    cfg.noise.add_noise = False\n    cfg.domain_rand.randomize_friction = False\n    cfg.domain_rand.randomize_base_mass = False\n    cfg.domain_rand.push_robots = False\n    cfg.terrain.mesh_type = 'plane'\n    cfg.terrain.curriculum = False\n    cfg.commands.curriculum = False\n    cfg.commands.heading_command = False  # 前進v, lateral=0, yaw rate=0を直接指定\n    cfg.commands.resampling_time = 1000000.0  # 評価中の指令更新を止める\n\n    env, _ = task_registry.make_env(name='g1', args=args, env_cfg=cfg)\n    runner, _ = task_registry.make_alg_runner(env=env, name='g1', args=args,\n                                               train_cfg=train_cfg, log_root=None)\n    model = Path(eval_args.eval_model)\n    assert model.is_file(), model\n    checkpoint = torch.load(str(model), map_location=env.device)\n    recorded_iter = checkpoint.get('iter')\n    print('Evaluate label:', eval_args.eval_label,\n          'Internal checkpoint iteration:', recorded_iter, flush=True)\n    assert 'model_state_dict' in checkpoint, 'model_state_dictが見つかりません．' \n    runner.alg.actor_critic.load_state_dict(checkpoint['model_state_dict'])\n    inference = runner.get_inference_policy(device=env.device)\n    # fresh Python process / fresh runner: LSTM hidden stateはNoneから開始\n    assert runner.alg.actor_critic.memory_a.hidden_states is None\n    device = env.device\n    dt = float(env.dt)\n    steps = int(round(duration / dt))\n    assert steps < env.max_episode_length\n\n    # 各速度をreps個並列に割り当てる．4方策とも同一seed，env順で評価．\n    targets = torch.tensor([v for v in speeds for _ in range(reps)],\n                           dtype=torch.float, device=device)\n    # runner初期化でenv.reset()が走っているので，その後に指令を書き換え，観測も再計算する\n    env.commands[:, :] = 0.0\n    env.commands[:, 0] = targets\n    env.compute_observations()\n    obs = env.get_observations()\n    assert obs.shape == (n, 47), obs.shape\n\n    survived = torch.ones(n, device=device, dtype=torch.bool)\n    first_fall = torch.full((n,), float('nan'), device=device)\n    sum_v = torch.zeros(n, device=device)\n    count_v = torch.zeros(n, device=device)\n    sum_v_steady = torch.zeros(n, device=device)\n    sum_e2_steady = torch.zeros(n, device=device)\n    count_steady = torch.zeros(n, device=device)\n    sum_roll2 = torch.zeros(n, device=device)\n    sum_pitch2 = torch.zeros(n, device=device)\n    min_height = torch.full((n,), float('inf'), device=device)\n    trajectory = []\n    contacts_log = []\n    forces_log = []\n    feet = env.feet_indices\n    assert feet.numel() == 2, feet\n    print('Feet indices (L/R):', feet.tolist(), flush=True)\n    previous_contact = (env.contact_forces[:, feet, 2] > 1.0).clone()\n    idx_reps = [k * reps for k in range(len(speeds))]\n\n    print('evaluation:', eval_args.eval_label, 'envs', n, 'steps', steps, 'dt', dt, flush=True)\n    with torch.no_grad():\n        for step in range(steps):\n            t = (step + 1) * dt\n            active = survived.clone()\n            actions = inference(obs.detach())\n            obs, _, _, dones, _ = env.step(actions.detach())\n            # 発生した終了は学習時の転倒条件（骨盤接触または姿勢閾値），タイムアウト前に限定\n            dones = dones.bool()\n            just_fell = active & dones\n            first_fall[just_fell] = t\n            survived &= ~dones\n            runner.alg.actor_critic.memory_a.reset(dones)\n\n            vx = env.base_lin_vel[:, 0]\n            roll = env.rpy[:, 0]\n            pitch = env.rpy[:, 1]\n            height = env.base_pos[:, 2]\n            alive_this_sample = active & (~dones)\n            val = alive_this_sample.float()\n            sum_v += torch.where(alive_this_sample, vx, torch.zeros_like(vx))\n            count_v += val\n            sum_roll2 += torch.where(alive_this_sample, roll ** 2, torch.zeros_like(roll))\n            sum_pitch2 += torch.where(alive_this_sample, pitch ** 2, torch.zeros_like(pitch))\n            min_height = torch.where(alive_this_sample, torch.minimum(min_height, height), min_height)\n            if t >= 6.0:\n                sum_v_steady += torch.where(alive_this_sample, vx, torch.zeros_like(vx))\n                sum_e2_steady += torch.where(alive_this_sample, (vx-targets)**2, torch.zeros_like(vx))\n                count_steady += val\n\n            # 左右接地変化（実験開始から）の記録．力の生値を閾値1 Nで接地判定\n            current_contact = env.contact_forces[:, feet, 2] > 1.0\n            if t >= 0.0:\n                changed = current_contact ^ previous_contact\n                for idx, side_idx in changed.nonzero(as_tuple=False).tolist():\n                    if bool(alive_this_sample[idx].item()):\n                        contacts_log.append({\n                            'model': eval_args.eval_label,\n                            'command_mps': speeds[idx // reps],\n                            'rep': idx % reps, 'side': ('L', 'R')[side_idx],\n                            'event': 'touchdown' if bool(current_contact[idx, side_idx]) else 'liftoff',\n                            'time_s': round(t, 5)\n                        })\n            # All surviving environments: left/right vertical contact force, dt~20ms\n            fz_l = env.contact_forces[:,feet[0],2].detach().cpu().tolist()\n            fz_r = env.contact_forces[:,feet[1],2].detach().cpu().tolist()\n            alive_mask = alive_this_sample.detach().cpu().tolist()\n            for j in range(n):\n                if alive_mask[j]:\n                    forces_log.append({'model':eval_args.eval_label,\n                                       'command_mps':speeds[j//reps], 'rep':j%reps,\n                                       'time_s':round(t,5), 'left_fz_N':fz_l[j],\n                                       'right_fz_N':fz_r[j]})\n            previous_contact = current_contact.clone()\n\n            # 各速度で最初の試行について，軌跡を取得．落ちた後は記録しない\n            for j in idx_reps:\n                if bool(alive_this_sample[j].item()):\n                    trajectory.append({\n                        'label': eval_args.eval_label, 'command_mps': float(targets[j].item()),\n                        'time_s': round(t, 5), 'base_vx_mps': float(vx[j].item()),\n                        'base_height_m': float(height[j].item()),\n                        'roll_rad': float(roll[j].item()),\n                        'pitch_rad': float(pitch[j].item())\n                    })\n            if not bool(survived.any().item()):\n                break\n\n    def cpu_np(x):\n        return x.detach().cpu().numpy()\n\n    ok = cpu_np(survived)\n    falls = cpu_np(first_fall)\n    obs_counts = cpu_np(count_v)\n    steady_counts = cpu_np(count_steady)\n    vmeans = cpu_np(sum_v)\n    vsteady = cpu_np(sum_v_steady)\n    e2 = cpu_np(sum_e2_steady)\n    r2 = cpu_np(sum_roll2)\n    p2 = cpu_np(sum_pitch2)\n    hmin = cpu_np(min_height)\n\n    rows = []\n    for j in range(n):\n        rows.append({\n            'label': eval_args.eval_label,\n            'command_mps': speeds[j // reps],\n            'rep': j % reps,\n            'seed': args.seed,\n            'survived_18s': int(ok[j]),\n            'fall_time_s': round(float(falls[j]), 4) if np.isfinite(falls[j]) else '',\n            'observed_s': round(float(obs_counts[j] * dt), 4),\n            'mean_vx_active_mps': float(vmeans[j] / obs_counts[j]) if obs_counts[j] else '',\n            'mean_vx_steady_mps': float(vsteady[j] / steady_counts[j]) if steady_counts[j] else '',\n            'rmse_vx_steady_mps': float(np.sqrt(e2[j] / steady_counts[j])) if steady_counts[j] else '',\n            'rms_roll_rad': float(np.sqrt(r2[j] / obs_counts[j])) if obs_counts[j] else '',\n            'rms_pitch_rad': float(np.sqrt(p2[j] / obs_counts[j])) if obs_counts[j] else '',\n            'min_base_height_m': float(hmin[j]) if np.isfinite(hmin[j]) else '',\n        })\n\n    dst = Path(eval_args.eval_out)\n    dst.mkdir(parents=True, exist_ok=True)\n    def save_csv(path, data):\n        if not data:\n            return\n        with path.open('w', newline='', encoding='utf-8') as f:\n            writer = csv.DictWriter(f, fieldnames=list(data[0]))\n            writer.writeheader()\n            writer.writerows(data)\n    save_csv(dst / f'trials_{eval_args.eval_label}.csv', rows)\n    save_csv(dst / f'trajectories_{eval_args.eval_label}.csv', trajectory)\n    save_csv(dst / f'foot_events_{eval_args.eval_label}.csv', contacts_log)\n    save_csv(dst / f'forces_{eval_args.eval_label}.csv', forces_log)\n    with (dst / f'config_{eval_args.eval_label}.json').open('w') as f:\n        json.dump({'model': str(model), 'label': eval_args.eval_label,\n                   'speeds_mps': speeds, 'reps_per_speed': reps,\n                   'duration_s': duration, 'seed': args.seed,\n                   'dt_s': dt,\n                   'evaluation': 'fixed x command / yaw=0 / no noise / no push / fixed friction and mass'}, f, indent=2)\n    print('saved:', dst, 'rows:', len(rows), 'survived:', int(np.sum(ok)), '/', n, flush=True)\n\n",str(EVAL_SCRIPT),'exec')
print('評価スクリプト:',EVAL_SCRIPT)

## 13．4方策の評価（18秒 × 3速度 × 各4試行）

In [ ]:
import subprocess,os,shutil
EVAL_DIR=EXP_DIR/'evaluation'
EVAL_DIR.mkdir(parents=True,exist_ok=True)
EVAL_MODELS={
  'baseline6500': SEED_MODEL,
  **{label: EXP_DIR/label/'checkpoints'/f'model_{FINAL_ITER}.pt' for label in TOLERANCES_MS},
}
for label,model in EVAL_MODELS.items():
    assert model.is_file(),model
    cmd=[str(PY38),str(EVAL_SCRIPT),'--task=g1','--headless',
         '--sim_device=cuda:0','--rl_device=cuda:0',
         '--num_envs=12','--seed=2026',
         '--eval_model',str(model),'--eval_label',label,
         '--eval_out',str(EVAL_DIR),'--eval_reps','4','--eval_duration','18']
    log=EVAL_DIR/f'eval_{label}.log'
    print('評価:',label,flush=True)
    with log.open('w',encoding='utf-8') as f:
        r=subprocess.run(cmd,cwd=str(repo),env=RUN_ENV,stdout=f,stderr=subprocess.STDOUT)
    print('終了コード:',r.returncode, '|', '\n'.join(log.read_text(encoding='utf-8',errors='replace').splitlines()[-9:]))
    assert r.returncode==0,log
print('P0/P1/P2と基準方策の推論評価が完了しました．')

## 14．非転倒率・速度追従・姿勢安定性

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
trials=[]
for label in EVAL_MODELS:
    a=pd.read_csv(EVAL_DIR/f'trials_{label}.csv')
    a['model']=label
    trials.append(a)
trials=pd.concat(trials,ignore_index=True)
summary=(trials.groupby(['model','command_mps'],as_index=False)
        .agg(n=('survived_18s','size'),survival=('survived_18s','mean'),
             steady_vx=('mean_vx_steady_mps','mean'),rmse=('rmse_vx_steady_mps','mean')))
summary.to_csv(EVAL_DIR/'P012_speed_survival_summary.csv',index=False)
print('速度追従と非転倒率'); display(summary.round(3))
fig,ax=plt.subplots(figsize=(7,4))
for label,g in summary.groupby('model'):
    ax.plot(g.command_mps,g.rmse,'o-',label=label)
ax.set(xlabel='Command velocity (m/s)',ylabel='Steady velocity RMSE (m/s)');ax.legend();ax.grid(alpha=.3)
plt.tight_layout();plt.savefig(EVAL_DIR/'P012_velocity_rmse.png',dpi=170);plt.show()

## 15．歩行周期，RL／LR時間，左右差の時間変化

着地時刻から，R→LをRL，L→RをLRとし，**LR−RL**（負ならRLの方が長い）を評価．
4〜18秒の定常区間のほか，初期0〜4秒を別に集計します．接地時刻の分解能は20 msです．

In [ ]:
# 接地時系列から周期とRL/LR時間を算出．時刻分解能は0.02 s，短時間チャタリングは除外
records=[]
for label in EVAL_MODELS:
    events=pd.read_csv(EVAL_DIR/f'foot_events_{label}.csv')
    for (speed,rep),z in events.groupby(['command_mps','rep']):
        td=z[z.event=='touchdown'].sort_values('time_s')
        filtered={}
        for side in ['L','R']:
            xx=td.loc[td.side==side,'time_s'].to_numpy()
            clean=[]
            for t in xx:
                if not clean or t-clean[-1]>=0.20: clean.append(t)
            filtered[side]=clean
            periods=np.diff(clean)
            periods=periods[(periods>=0.35)&(periods<=1.5)]
            for p in periods:
                records.append({'model':label,'command_mps':speed,'rep':rep,
                                'metric':f'{side}_period','duration_ms':p*1000,'start_s':float('nan')})
        merged=sorted([(t,side) for side,zz in filtered.items() for t in zz])
        for (t0,s0),(t1,s1) in zip(merged[:-1],merged[1:]):
            d=t1-t0
            if s0!=s1 and .20<=d<=.65:
                metric='RL' if s0=='R' else 'LR'
                records.append({'model':label,'command_mps':speed,'rep':rep,
                                'metric':metric,'duration_ms':d*1000,'start_s':t0})
intervals=pd.DataFrame.from_records(records)
assert not intervals.empty,'着地イベントが取得できません．foot_eventsの記録を確認してください．'
intervals.to_csv(EVAL_DIR/'gait_intervals.csv',index=False)
gait=(intervals.groupby(['model','command_mps','metric'])['duration_ms']
      .agg(['count','mean','std']).round(2).reset_index())
print('全区間の周期・RL/LR間隔（ms）');display(gait)
for metric in ['L_period','R_period','RL','LR']:
    subset=gait[gait.metric==metric]
    fig,ax=plt.subplots(figsize=(6.5,3.5))
    for label,g in subset.groupby('model'):
        ax.plot(g.command_mps,g['mean'],'o-',label=label)
    ax.set(xlabel='Command vx (m/s)',ylabel=f'{metric} (ms)',title=metric)
    ax.grid(alpha=.3);ax.legend();plt.tight_layout()
    fig.savefig(EVAL_DIR/f'P012_{metric}.png',dpi=170);plt.show()

In [ ]:
# 定常区間と初期区間を分離して集計（RL/LR間隔）
rl=intervals[intervals.metric.isin(['RL','LR'])].copy()
rl['window']=np.where(rl.start_s < 4.0, 'startup_0_4s', 'steady_4_18s')
window_summary=(rl.groupby(['model','command_mps','window','metric'])['duration_ms']
                  .agg(['count','mean','std']).reset_index())
window_summary.to_csv(EVAL_DIR/'P012_RL_LR_by_time_window.csv',index=False)
display(window_summary.round(2))
# 絶対左右差は，R→L と直後の L→R の連続2間隔で構成する
paired=[]
for label in EVAL_MODELS:
    for (speed,rep), z in rl[rl.model==label].groupby(['command_mps','rep']):
        z=z.sort_values('start_s')
        x=z.to_dict('records')
        for a,b in zip(x[:-1],x[1:]):
            if a['metric']=='RL' and b['metric']=='LR' and abs((a['start_s']+a['duration_ms']/1000)-b['start_s'])<=0.021:
                paired.append({'model':label,'command_mps':speed,'rep':rep,
                               'start_s':a['start_s'],
                               'RL_ms':a['duration_ms'],'LR_ms':b['duration_ms'],
                               'LR_minus_RL_ms':b['duration_ms']-a['duration_ms'],
                               'abs_LR_minus_RL_ms':abs(b['duration_ms']-a['duration_ms'])})
paired=pd.DataFrame(paired)
assert not paired.empty,'完全な左右接地間隔ペアが検出できません'
paired.to_csv(EVAL_DIR/'P012_RL_LR_paired_cycles.csv',index=False)
steady=paired[paired.start_s>=4].groupby(['model','command_mps']).agg(
    n=('LR_ms','size'),RL_ms=('RL_ms','mean'),LR_ms=('LR_ms','mean'),
    asym_ms=('LR_minus_RL_ms','mean'),abs_asym_ms=('abs_LR_minus_RL_ms','mean')).reset_index()
display(steady.round(2))
steady.to_csv(EVAL_DIR/'P012_RL_LR_steady_comparison.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4))
for label,g in steady.groupby('model'):
    ax.plot(g.command_mps,g.asym_ms,'o-',label=label)
ax.axhline(0,c='gray',lw=1)
ax.set(xlabel='Command vx (m/s)',ylabel='LR-RL (ms)',title='Steady asymmetry (4-18s)')
ax.legend();ax.grid(alpha=.3);plt.tight_layout()
fig.savefig(EVAL_DIR/'P012_steady_asymmetry.png',dpi=170);plt.show()
for speed in [0.2,0.6,1.0]:
    fig,ax=plt.subplots(figsize=(9,3.5))
    for label,g in paired[paired.command_mps==speed].groupby('model'):
        # descriptive time-course: mean across repeats in 1s bins
        bins=g.assign(tbin=np.floor(g.start_s).astype(int)).groupby('tbin').LR_minus_RL_ms.mean()
        ax.plot(bins.index,bins.values,'o-',label=label)
    ax.axhline(0,c='gray',lw=1)
    ax.axvline(4,c='black',lw=1,ls='--',alpha=.5)
    ax.set(xlabel='Time bin (s)',ylabel='LR-RL (ms)',title=f'Asymmetry over time, cmd={speed} m/s')
    ax.grid(alpha=.3);ax.legend();plt.tight_layout()
    fig.savefig(EVAL_DIR/f'P012_asymmetry_time_{speed}.png',dpi=170);plt.show()

## 16．左右接地時間・鉛直接触力（保存・補助解析）

接地は鉛直接触力1 N超で判定．記録間隔20 msなので**着地衝撃の真の最大値ではなくサンプル最大値**です．
「接地中平均力」と「非接地時間も含む時間平均」を混同しないため，ここでは1接地ごとの**接地中平均力，サンプルピーク力，力積**を算出します．

In [ ]:
# For each model/speed/rep/foot, pair touchdown to subsequent liftoff;
# force samples in [touchdown, liftoff), valid events after 4s.
stance_rows=[]
for label in EVAL_MODELS:
    events=pd.read_csv(EVAL_DIR/f'foot_events_{label}.csv')
    traces=pd.read_csv(EVAL_DIR/f'forces_{label}.csv')
    for (speed,rep),evt in events.groupby(['command_mps','rep']):
        f=traces[(traces.command_mps==speed)&(traces.rep==rep)].sort_values('time_s')
        for side in ['L','R']:
            z=evt[evt.side==side].sort_values('time_s')
            td=z[z.event=='touchdown'].time_s.to_numpy()
            lo=z[z.event=='liftoff'].time_s.to_numpy()
            field='left_fz_N' if side=='L' else 'right_fz_N'
            for t0 in td:
                future=lo[lo>t0]
                if not len(future):continue
                t1=float(future[0])
                if t0<4 or t1>18.0 or not (0.18<=t1-t0<=0.7):continue
                ff=f[(f.time_s>=t0-1e-6)&(f.time_s<t1-1e-6)][field].to_numpy()
                if len(ff)<5:continue
                stance_rows.append({'model':label,'command_mps':speed,'rep':rep,'side':side,
                    'touchdown_s':t0,'liftoff_s':t1,'stance_ms':(t1-t0)*1000,
                    'contact_mean_N':float(np.mean(ff)),'recorded_peak_N':float(np.max(ff)),
                    'impulse_Ns':float(np.sum(ff)*0.02)})
stance=pd.DataFrame(stance_rows)
assert not stance.empty,'接地時間・力のCSVが読み取れません'
stance.to_csv(EVAL_DIR/'P012_per_stance_force.csv',index=False)
fs=(stance.groupby(['model','command_mps','side'])
    .agg(n=('stance_ms','size'),stance_ms=('stance_ms','mean'),
         contact_mean_N=('contact_mean_N','mean'),
         recorded_peak_N=('recorded_peak_N','mean'),
         impulse_Ns=('impulse_Ns','mean')).reset_index())
fs.to_csv(EVAL_DIR/'P012_contact_force_summary.csv',index=False)
display(fs.round(2))

## 17．結果の解釈と次の判断

1. **第一評価**：転倒率と速度追従が著しく悪化していないか．
2. **主要評価**：RL−LRの符号と絶対左右差がP0→P1→P2でどう変わるか．起動後の**時間変化**にも注目．
3. **副次評価**：歩行周期，接地時間，鉛直接触力，力積．
4. 20 ms分解能では10 ms程度の差を精密に解釈できません．4試行は同一学習・評価シードであり独立標本ではありません．
5. P0・P1・P2は**共通の6500反復学習済み方策からの200反復の追加学習**です．許容幅が有効かを厳密に判断するには，複数学習シード・追加学習長の検討が必要です．
6. 今回は**接地報酬の許容幅**の比較です．位相入力と周期0.8秒は固定のままです．

Notebookの学習・評価用スクリプトは構文と数式の単体テストを含みますが，Colabでの実GPU実行は利用者側で行ってください．